<a href="https://colab.research.google.com/github/ayush55223311/Ayush-Raj/blob/main/predictive_maintenance_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Overheat: Predictive Maintenance AI
**AeroLogistics — Data Science Division**

**Intern Name:** Ayush Raj  
**Course:** Data Science Internship  
**Project:** Predictive Maintenance AI

## Executive Summary
This notebook builds a time-series machine-learning workflow to flag catastrophic motor failure **24 hours ahead** using IoT telemetry. It engineers rolling sensor features, shifts the target forward in time, uses a chronological 80/20 split to avoid future-data leakage, trains a class-balanced Random Forest, and explains its predictions using feature importance.

**Data note:** The provided IoT dataset is embedded as a compressed fallback so this notebook can run as a single `.ipynb` file. If a local CSV named `iot_sensor_telemetry.csv` or `iot_sensor_telemetry (1).csv` is available, the notebook uses that file first.

In [ ]:
# 0. ENVIRONMENT SETUP & IMPORTS
import os
import io
import gzip
import base64
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

---
## Phase 1: Feature Engineering (Rolling Windows)

1. Parse timestamps and sort chronologically.
2. Calculate the 12-hour rolling mean of temperature.
3. Calculate the 12-hour rolling standard deviation of vibration.
4. Drop the initial rows with NaN values created by rolling calculations.

In [ ]:
# 1. LOAD DATA & ENGINEER FEATURES

# Prefer a supplied CSV if present; otherwise use the embedded copy to keep this notebook self-contained.
candidate_files = [
    "iot_sensor_telemetry.csv",
    "iot_sensor_telemetry (1).csv",
    "/mnt/data/iot_sensor_telemetry (1).csv",
]
csv_file = next((path for path in candidate_files if os.path.exists(path)), None)

if csv_file:
    df = pd.read_csv(csv_file)
    print(f"Loaded telemetry from: {csv_file}")
else:
    # Embedded compressed CSV fallback (keeps the deliverable to one notebook file).
    embedded_csv_gzip_b64 = '...'
    raw_csv = gzip.decompress(base64.b64decode(embedded_csv_gzip_b64))
    df = pd.read_csv(io.BytesIO(raw_csv))
    print("Loaded the embedded IoT telemetry dataset.")

required_columns = [
    "Timestamp", "Sensor_Temp_C", "Sensor_Vibration_mm",
    "Sensor_Voltage_V", "Catastrophic_Failure"
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="raise")
df = df.sort_values("Timestamp").reset_index(drop=True)

# A 12-row rolling window represents 12 hours because the data is hourly.
df["Temp_Rolling_Mean_12h"] = df["Sensor_Temp_C"].rolling(window=12, min_periods=12).mean()
df["Vibration_Rolling_Std_12h"] = df["Sensor_Vibration_mm"].rolling(window=12, min_periods=12).std()

rows_before = len(df)
df = df.dropna(subset=["Temp_Rolling_Mean_12h", "Vibration_Rolling_Std_12h"]).copy()
print(f"Rows removed due to rolling-window NaNs: {rows_before - len(df)}")
print(f"Phase 1 complete. Shape after feature engineering: {df.shape}")
display(df.head())

---
## Phase 2: Target Shifting (24-hour Lead Time)

`shift(-24)` places the failure label from 24 rows in the future onto the current row. Since the telemetry is hourly and sorted, the model learns to predict a failure 24 hours ahead. The original failure label is removed from the features to prevent target leakage.

In [ ]:
# 2. SHIFT THE TARGET VARIABLE

df["Failure_In_24_Hours"] = df["Catastrophic_Failure"].shift(-24)

# The final 24 rows have no known target 24 hours ahead, so remove them.
rows_before_shift_drop = len(df)
df = df.dropna(subset=["Failure_In_24_Hours"]).copy()
df["Failure_In_24_Hours"] = df["Failure_In_24_Hours"].astype(int)

# Remove original event label and timestamp from model inputs.
df = df.drop(columns=["Catastrophic_Failure", "Timestamp"])

print(f"Rows removed because future target is unavailable: {rows_before_shift_drop - len(df)}")
print("Phase 2 complete. Target shifted successfully.")
print("Target counts (0 = no failure in 24h, 1 = failure in 24h):")
print(df["Failure_In_24_Hours"].value_counts().sort_index())

---
## Phase 3: Chronological Train/Test Split

The first 80% of the timeline is used for training and the final 20% for testing. No random shuffling is performed, avoiding time-series leakage.

In [ ]:
# 3. CHRONOLOGICAL TRAIN/TEST SPLIT

target_col = "Failure_In_24_Hours"
feature_cols = [col for col in df.columns if col != target_col]
X = df[feature_cols].copy()
y = df[target_col].copy()

split_index = int(len(df) * 0.8)
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(f"Phase 3 complete. Training rows: {len(X_train)} | Test rows: {len(X_test)}")
print(f"Positive failure labels — train: {int(y_train.sum())}, test: {int(y_test.sum())}")
print("Features:", feature_cols)

NameError: name 'df' is not defined

---
## Phase 4: Model Training & Evaluation

Train a Random Forest with `class_weight="balanced"` because failure events are rare. The final test period remains untouched during training. Since failures are extremely rare, inspect the confusion matrix and failure-class recall, not accuracy alone.

In [ ]:
# 4. TRAIN THE PREDICTIVE MODEL

model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Default 0.5 probability threshold; keep this explicit for transparency.
y_pred = model.predict(X_test)

print("Classification Report (predicting 24h in advance):")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["No failure in 24h (0)", "Failure in 24h (1)"],
    zero_division=0
))

print("Confusion Matrix (rows = actual, columns = predicted):")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No failure", "Failure in 24h"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Chronological Test Period")
plt.tight_layout()
plt.show()

print(f"Failure-class precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Failure-class recall:    {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Failure-class F1-score:  {f1_score(y_test, y_pred, zero_division=0):.3f}")

if int(y_test.sum()) == 0:
    print("Warning: The test period contains no positive failure events, so failure recall cannot be meaningfully evaluated.")
elif recall_score(y_test, y_pred, zero_division=0) == 0:
    print("Warning: The model did not detect any test-period failures. This can happen with very rare events; accuracy alone would be misleading.")

NameError: name 'RandomForestClassifier' is not defined

---
## Phase 5: Explainable AI & Business Strategy

Rank engineered/input features by Random Forest feature importance. These scores describe how much the trained forest used each feature; they do not prove causation.

In [ ]:
# 5. FEATURE IMPORTANCE (EXPLAINABLE AI)

importances = model.feature_importances_
feature_importance = pd.Series(importances, index=X.columns).sort_values(ascending=True)

print("Feature importances (ascending):")
display(feature_importance.to_frame("Importance"))

ax = feature_importance.plot.barh(color="steelblue", figsize=(9, 5))
ax.set_title("Predictive Maintenance — Feature Importance")
ax.set_xlabel("Importance Score")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

ranked_features = feature_importance.sort_values(ascending=False)
print("Top feature:", ranked_features.index[0], f"({ranked_features.iloc[0]:.3f})")
if len(ranked_features) > 1:
    print("Second feature:", ranked_features.index[1], f"({ranked_features.iloc[1]:.3f})")

NameError: name 'model' is not defined

---
## Final Executive Recommendation

Use the feature-importance ranking above to guide sensor monitoring priorities. Prioritize the highest-ranked feature and the second-highest feature for closer inspection, while continuing to monitor all sensors. Treat this model as a decision-support alert rather than an automatic replacement decision.

**Important operational limitation:** the dataset contains very few catastrophic-failure events. The final chronological test set may contain only a handful of failures, so recall estimates are unstable. Before deployment, validate the model on additional failure examples, review missed failures with maintenance engineers, and choose an alert threshold based on the acceptable trade-off between missed failures and false alarms.

**Deployment status:** Prototype only — further validation required.